In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, roc_curve, auc, precision_recall_curve

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

# Memory optimization
tf.keras.backend.clear_session()

In [ ]:
df_normal = pd.read_csv("file_name", low_memory = True)
df_normal.columns = df_normal.columns.str.strip() # delete space 

df_normal.head()

In [ ]:
# feature engineering - optional 
sensor_patterns = ['Temp', 'Peak'] # name of all extracted features 
sensor_cols = [c for c in df_normal.columns if any(c.startswith(p) for p in sensor_patterns)]
# lấy các cột sensor 


def add_features(df_normal, sensor_cols):
    new_cols = []
    
    key_sensors = sensor_cols[:6]  # assuming more than 6 features from 1 sensor
    
    for sensor in key_sensors: 
        col = f"{sensor}_rm10"
        df_normal[col] = df_normal[sensor].rolling(10, min_periods = 1).mean().astype(np.float32)
        # can adjust window size more or less than 10 
        new_cols.append(col)
        col = f"{sensor}_diff"
        df_normal[col] = df[sensor].diff().fillna(0).astype(np.float32)
        new_cols.append(col)
    return new_cols

new_features = add_features(df_normal, sensor_cols)
modeling_features = sensor_cols + new_features # all are numerical features 
print(f"Total modeling features: {len(modeling_features)}")
gc,collect()

In [ ]:
# lưu scaler đã fit
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(df_normal).astype(np.float32)

joblib.dump(scaler, "preprocessor.joblib")

In [ ]:
print("Time for training with Isolation Forest")
iso_forest = IsolationForest(n_estimators = 100, contamination = 0.05, random_state = 42,
                            n_job = -1)

iso_forest.fit(X_train_scaled)
joblib.dump(iso_forest, "isolation_forest_model.joblib")

In [ ]:
y_pred = iso_forest.predict(X_train_scaled)

print("Abnormal values: ", np.sum(y_pred == -1))
print('Normal values: ', np.sum(y_pred == 1))

abnormal_indices = np.where(y_pred == -1)[0][:5]
print("Example of abnormal values: ", X_train_scaled[abnormal_indices])

# clear_result = iso_forest.decision_function(X_train_scaled)
# print("Abnormal values: ", np.sum(clear_result > -0.5))
# clear_result_indices = np.where(clear_result > -0.5)[0][:5]
# print("Example of abnormal values: ", X_train_scaled[clear_result_indices])

In [ ]:
print("Time for One class SVM")
ocsvm = OneClassSVM(kernel = 'rbf', nu = 0.05, gamma = 'scale')
ocsvm.fit(X_train_scaled)

joblib.dump(ocsvm, "oneclass_svm.joblib")

In [ ]:
y1_pred = ocsvm.predict(X_train_scaled)

print("Abnormal values: ", np.sum(y1_pred == -1))
print("Normal values: ", np.sum(y1_pred == 1))

abnormal_indices1 = np.where(y1_pred == -1)[0][:5]
print("Example of abnormal values: ", abnormal_indices)

# clear_result1 = ocsvm.decision_function(X_train_scaled)
# print("Abnormal values: ", np.sum(clear_result1 > -0.5))
# clear_result_indices1 = np.where(clear_result1 > -0.5)[0][:5]
# print("Example of abnormal values: ", X_train_scaled[clear_result_indices1])

In [ ]:
SEQ_LENGTH = 5 
n_features = X_train_scaled.shape[1]

def create_sequences_efficient(data, seq_length):
    n = len(data) - seq_length + 1
    sequences = np.zeros((n, seq_length, data.shape[1]), dtype = np.float32)
    for i in range(n): 
        sequence[i] = data[i:i + seq_length]
    return sequences 

X_train_seq = create_sequences_efficient(X_train_scaled, SEQ_LENGTH)

print(f"Train sequence: {X_train_seq.shape}")
gc.collect()

In [ ]:
tf.keras.backend.clear_session() 
def build_lstm_ae(seq_length, n_features):
    inputs = keras.Input(shape = (seq_length, n_features))
    x = layer.LSTM(32, activation = 'relu', return_sequences = False)(inputs)
    encoded = layers.Dense(16, activation = 'relu')(x)
    
    x = layers.RepeatVector(seq_length(encoded ))
    
    x = layers.LSTM(32, activation = 'relu', return_sequences = True)(x)
    outputs = layers.TimeDistributed(layers.Dense(n_features))(x)
    model = Model(inputs, outputs)
    model.compile(optimizer = 'adam', loss = 'mse')
    return model 

lstm_ae = build_lstm_ae(SEQ_LENGTH, n_features)
lstm_ae.summary()

In [ ]:
print("Training LSTM-Autoencoder")
history = lstm_ae.fit(
    X_train_seq, X_train_seq, 
    epochs = 20, batch_size = 128,
    validation_split = 0.1,
    callbacks = [EarlyStopping(patience = 3, restore_best_weights = True)], verbose = 1
)

joblib.dump(lstm_ae, "lstm_autoencoder.h5")

In [ ]:
plt.figure(figsize = (10,4))
plt.plot(hsitory.history['loss'], label = 'Train')
plt.plot(history.history['val_loss'], label = 'Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('LSTM Autoencoder Training')
plt.show()

In [ ]:
print("Calculating reconstruction error ")
X_train_pred = lstm_ae.predict(X_train_seq, batch_size = 256, verbose = 0)
mse_train = np.mean(np.square(X_train_seq - X_train_pred), axis = (1,2))

threshold_95 = np.percentile(mse_train, 95)
threshold_99 = np.percentile(mse_train, 99)

print(f"Threshold at 95%: {threshold_95:.6f}")
print(f"Threshold at 99%: {threshold_99:.6f}")

In [ ]:
# Reconstruction error distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(mse_train, bins=80, alpha=0.6, density=True)
ax.axvline(threshold_95, color='orange', linestyle='--', lw=2, label=f'95th: {threshold_95:.3f}')
ax.axvline(threshold_99, color='red', linestyle='--', lw=2, label=f'99th: {threshold_99:.3f}')
ax.set_xlabel('Reconstruction Error')
ax.set_ylabel('Density')
ax.set_xlim(0, np.percentile(mse_test, 99)) # 99% giá trị abnormal nằm dưới ngưỡng này 
ax.legend()
ax.set_title('Reconstruction Error Distribution')
plt.show()

In [ ]:
# This cell only use for plotting after fit the new dataset. 

df_new = pd.read_csv("name of new dataset file after transformed", low_memory = True)
X_test_seq = create_sequences_efficient(df_new, SEQ_LENGTH)

sample_idx = 5
original = X_train_seq[sample_idx]
reconstructed = X_test_pred[sample_idx]

n_features_to_plot = 3 

plt.figure(figsize = (12,6))

for i in range(n_features_to_plot):
    plt.subplot(n_features_to_plot, 1, i+1)   # i is which index at 
    plt.plot(original[:, i], label="Original", alpha = 0.8)
    plt.plot(reconstructed[:, i], label = "Reconstructed", alpha = 0.8)
    plt.title(f"Feature{i+1}")
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show() 